# Introduction

Date: 20250219
- The purpose of this notebook is to create an h5ad file that contains the integrated spatial and scRNA object as well as add important metadata.

In [ ]:
import pickle
import os
import itertools
import scipy
import csv
import pandas as pd
import numpy as np
import scanpy as sc
import scvelo as scv
import gseapy as gp
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.pyplot import rc_context
from matplotlib.colors import LogNorm, Normalize
from scipy import sparse
from scipy import stats
from copy import copy
from scipy.stats import norm
from sklearn.neighbors import KernelDensity
from typing import Tuple
from pathlib import Path
from scipy.io import mmread
from matplotlib.colors import LinearSegmentedColormap, Normalize

# Load helpful objects and functions

In [2]:
all_datasets = ['E11_iScience_R1', 'E12_DevCell', 'E12_CellReports_R1', 'E13_CellReports', 'E12_Science', 'E11lungR1', 'E11lungR2', 'E11lungR3', 'E11lungR4', 'E11lungR5', 'E12lungR2', 'E12lungR3', 'E12lungR4', 'E12lungR5', 'E13lungR1', 'E13lungR2', 'E13lungR3', 'E13lungR4']
dbit_datasets = ['E11lungR1', 'E11lungR2', 'E11lungR3', 'E11lungR4', 'E11lungR5', 'E12lungR2', 'E12lungR3', 'E12lungR4', 'E12lungR5', 'E13lungR1', 'E13lungR2', 'E13lungR3', 'E13lungR4']
ref_datasets = ['E11_iScience_R1', 'E12_DevCell', 'E12_CellReports_R1', 'E13_CellReports', 'E12_Science']
timepoints = ['E11.5', 'E12.5', 'E13.5']
cellstates = ['MesenchymeProgenitor', 'SmoothMuscleProgenitor', 'SmoothMuscle', 'MatrixMesenchyme', 'Precartilage', 'Pericyte',
              'Mesothelium', 'Epithelial(Distal)', 'Epithelial(Proximal)', 'VascularEndothelial',
              'Immune', 'Blood', 'Neuron', 'Heart']
epi_subcluster_cellstates = ['Distal', 'DistalIntermediate', 'ProximalIntermediate', 'Proximal']
mes_subcluster_cellstates = ['MP1', 'MP2', 'MP3', 'SmoothMuscleProgenitor', 'SmoothMuscle', 'MatrixMesenchyme', 'MatrixMesenchymeProgenitor', 'Precartilage', 'Pericyte']

# figure out color scheme
clusterColors = {'Neuron': 'DarkGreen',
                 'MatrixMesenchyme': 'DarkRed',
                 'MesenchymeProgenitor': 'RoyalBlue',
                 'Epithelial(Distal)': 'SkyBlue',
                 'Epithelial(Proximal)': 'Red',
                 'DbitEpithelialBridge': 'Cyan',
                 'SmoothMuscleProgenitor': 'Magenta',
                 'SmoothMuscle': 'Purple',
                 'VascularEndothelial': 'gold',
                 'Mesothelium': 'limegreen',
                 'Precartilage': 'darkgoldenrod',
                 'Immune': 'DarkOrange',
                 'Heart': 'Gray',
                 'Blood': 'tan',
                 'Pericyte': 'lightpink'}

epiSubclusterColors = {'Proximal': 'Red',
                    'ProximalIntermediate':'darkred',
                    'DistalIntermediate':'darkblue',
                    'Distal':'SkyBlue'}

mesSubclusterColors = {'MP1': 'RoyalBlue',
                       'MP2':'lawngreen',
                       'MP3':'Orange',
                       'MP4':'salmon',
                       'MatrixMesenchyme':'DarkRed',
                       'MatrixMesenchymeProgenitor': 'Red',
                       'SmoothMuscleProgenitor': 'Magenta',
                       'SmoothMuscle':'Purple',
                       'Pericyte': 'lightpink',
                       'Precartilage':'darkgoldenrod'}

phaseDict = {'G1': 'Blue', 'G2M': 'Orange', 'S': 'Green'}

greens = copy(mpl.cm.Greens)
greens.set_under("lightgray")
reds = copy(mpl.cm.Reds(np.linspace(0,1,40)))
reds = mpl.colors.ListedColormap(reds[5:])
reds.set_under("lightgray")

# Add silver as the lower color for the expression colormap
colors_1 = ["silver", "red"]  # Colors for the gradient
expr_cmap = LinearSegmentedColormap.from_list("silver_to_red", colors_1)
expr_cmap.set_over("red") 
norm = Normalize(vmin=0, vmax=10)

In [3]:
def plot_spatial_groups(adata, 
                        meta_col,
                        background_adata = False, 
                        palette = False, 
                        show = True, 
                        output_loc = False, 
                        name = 'spatial_plot', 
                        size = 20):
    '''
    input
        - adata: An anndata object. Assumes 14 dbit datasets (split by 'orig.ident'), labeled as dbit in the 'tech' column. Assumes there is an x_loc and y_loc col
        - background_adata: A second anndata object that contains all the cells to plot in the background (if adata is a subset).
            if not provided then it will take adata as the background adata.
        - meta_col: A column in the .obs table or gene in the .var table
        - palette: A dictionary that contains a desired color for each value in the meta_col column or a cmap for a quantitative meta_col.
        - show: True or False on whether to show the plot. Default is False
        - output_loc: Whether or not to save the plot and what folder location. Default is False. 
        - name: The name to give the output pdf if saving the plot
        - size: The dot size for the scatter plot
    returns
        creates spatial scatter plots for the 14 lung spatial datasets. Options to save and show the plot
    '''
    # check if the output_loc folder exists, return an error if not
    if output_loc != False:
        if not os.path.exists(output_loc):
            print('Error: output_loc folder does not exist')
            return
        
    # check if meta_col exists, returns an error if not
    if meta_col not in adata.obs.columns and meta_col not in adata.var.index:
        print('Error: meta_col not in .obs or .var tables')
        return
    
    if type(background_adata) == type(False):
        background_adata = adata
    
    # Make a temp object to work with
    temp_adata = adata.copy()
    
    # Add the gene expr values to the obs table if plotting a gene
    if meta_col in adata.var.index:
        temp_adata.obs[meta_col] = temp_adata[:, meta_col].X.todense()
    
    timepoints = ['E11.5', 'E12.5', 'E13.5']
    fig, axs = plt.subplots(3, 5, figsize = (35, 21))
    row_count = 0
    for time in timepoints:
        datasets = temp_adata.obs[(temp_adata.obs['timepoint'] == time) & (temp_adata.obs['tech'] == 'DBiT_seq')]['orig.ident'].unique()

        col_count = 0
        for exp in datasets:
            temp_ax = axs[row_count, col_count]
            col_count += 1
            
            # Plot full structure in gray from the background adata
            all_cell_locs = background_adata.obs[(background_adata.obs['orig.ident'] == exp)].index
            x = []
            y = []

            for i in all_cell_locs:
                y.append(background_adata.obs.loc[i, 'y_loc'])
                x.append(background_adata.obs.loc[i, 'x_loc'])
            # Use silver with a default alpha to match the lower bound of the expression plots
            temp_ax.scatter(x, y, color = 'silver', s = size)
        
            # plot the quantitative variable
            temp_locs = temp_adata.obs[(temp_adata.obs['orig.ident'] == exp)].index
            
            if type(temp_adata.obs[meta_col][0]) != str:
                x = [0]
                y = [0]
                z = [temp_adata.obs[temp_adata.obs['tech'] == 'DBiT_seq'][meta_col].max()]
                for i in temp_locs:
                    z.append(temp_adata.obs.loc[i, meta_col])
                    y.append(temp_adata.obs.loc[i, 'y_loc'])
                    x.append(temp_adata.obs.loc[i, 'x_loc'])
                if palette != False:
                    colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = palette)
                else:
                    colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = 'viridis')
            else:
                labels = ['background']
                state_list = temp_adata.obs[meta_col].unique().tolist()
                for state in state_list:
                    temp_locs = temp_adata.obs[(temp_adata.obs['orig.ident'] == exp) & (temp_adata.obs[meta_col] == state)].index

                    x = [0]
                    y = [0]
                    s = [size]
                    for i in temp_locs:
                        y.append(temp_adata.obs.loc[i, 'y_loc'])
                        x.append(temp_adata.obs.loc[i, 'x_loc'])
                        s.append(size)
                    labels.append(state)

                    if palette != False:
                        temp_ax.scatter(x, y, s = size, color = palette[state])
                    else:
                        temp_ax.scatter(x, y, s = size)
                    if row_count == 0 and col_count == 5:
                        temp_ax.legend(labels = labels, bbox_to_anchor = (1.02, 1))
            temp_ax.set_title('{}_{}'.format(exp, meta_col))
            temp_ax.set_xlabel('X Pos')
            temp_ax.set_ylabel('Y Pos')
            temp_ax.set_xlim(0, 52)
            temp_ax.set_ylim(0, 52)

        row_count += 1
    if type(temp_adata.obs[meta_col][0]) != str:
        fig.colorbar(colorbar, ax = axs[2, 4])
    if output_loc != False:
        plt.savefig(output_loc + name, dpi = 300)
    if show:
        plt.show()
    plt.close()

# Create a new dataset that contains the raw and normalized counts

In [11]:
# This is the sct output dataset
inte_adata = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/integrated/grcm36/Lung_dbit_srf_CellReports_Science_integrated.h5ad')

# Read in the raw count mtx from the seurat integrated object
raw_mtx = mmread('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/integrated/grcm36/Lung_Integrated_raw_counts.mtx')
sct_mtx = mmread('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/integrated/grcm36/Lung_Integrated_SCT_counts.mtx')
genes = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/integrated/grcm36/Lung_Integrated_raw_counts_genes.csv')
SCT_genes = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/integrated/grcm36/Lung_Integrated_raw_counts_SCT_genes.csv', index_col = 0)
cells = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/integrated/grcm36/Lung_Integrated_raw_counts_cells.csv')
metadata = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/integrated/full_dataset/Lung_Integrated_metadata.csv', index_col = 0, low_memory = False)
umap = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/integrated/full_dataset/Lung_Integrated_umap.csv', index_col = 0)

# clean up the genes tables
genes['Raw'] = genes['x']
SCT_genes['SCT'] = SCT_genes['x']
genes.set_index('x', inplace = True)
SCT_genes.set_index('x', inplace = True)
genes.drop(columns = 'Unnamed: 0', inplace = True)

# Create a new adata that is made up of the raw counts from the seurat object, copy over metadata and reductions
lung_adata = ad.AnnData(raw_mtx.T)
lung_adata.obs = inte_adata.obs
lung_adata.uns = inte_adata.uns
lung_adata.obsm = inte_adata.obsm
lung_adata.var = genes
lung_adata.obsp = inte_adata.obsp
lung_adata.X = scipy.sparse.csr_matrix(lung_adata.X.todense())

# Drop the genes that aren't in the SCT mtx, and add SCT to a layer
drop_genes = []
for gene in lung_adata.var.index:
    if not gene in SCT_genes.index:
        drop_genes.append(gene)
gene_indicator = np.in1d(lung_adata.var_names, drop_genes)
lung_adata = lung_adata[:, ~gene_indicator]

# save the raw counts before normalization and logscaling
lung_adata.layers['Raw_Counts'] = lung_adata.X
lung_adata.layers['SCT'] = scipy.sparse.csr_matrix(sct_mtx.T.todense())

In [12]:
lung_adata.obs['tech'] = lung_adata.obs['tech'].astype('str')

for i in lung_adata.obs.index:
    if lung_adata.obs.loc[i, 'tech'] != 'DBiT_seq':
        lung_adata.obs.loc[i, 'tech'] = 'scRNA_seq'

In [14]:
# clean up the object metadata
lung_adata.obs.drop(columns = ['old.orig.ident', 'SCT_snn_res.0.5', 'SCT_snn_res.0.6'], inplace = True)

# Normalize and log the top layer

In [20]:
# Normalize, logscale, and scale the 
sc.pp.normalize_total(lung_adata, target_sum=1e4)
sc.pp.log1p(lung_adata)

In [9]:
# Convert ensambl IDs back into gene names

# Build dictionary for converting GeneID to Gene names
file_path_gene = '/Genomics/chanlab/pengfei/dbit/Manuscript/lungmapping/computation manual/RNAvelocity/Ensembl_geneID_conversion_v2.txt'

# Create an empty dictionary to store the data
data_dict_gene = {}

# Open the file and read its contents
with open(file_path_gene, 'r') as file:
    for line in file:
        # Split each line into key and value using the tab separator
        key, value = line.strip().split(',')
        
        # Add key-value pairs to the dictionary
        data_dict_gene[key] = value
flipped_gene_dict = {value: key for key, value in data_dict_gene.items()}

# Build dictionary for converting GeneID to Gene names using the v3 table
file_path_gene_v3 = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/Ensembl_geneID_conversion_v3.txt'

# Create an empty dictionary to store the data
data_dict_gene_v3 = {}

# Open the file and read its contents
with open(file_path_gene_v3, 'r') as file:
    for line in file:
        # Split each line into key and value using the tab separator
        value, key = line.strip().split('\t')
        
        # Add key-value pairs to the dictionary
        data_dict_gene_v3[key] = value
flipped_gene_dict_v3 = {value: key for key, value in data_dict_gene.items()}

In [22]:
lung_adata.var['ensamblID'] = lung_adata.var.index

count = 0
bad = 0
for i in lung_adata.var.index:
    if i in data_dict_gene_v3.keys():
        lung_adata.var.loc[i, 'gene'] = data_dict_gene_v3[i]
    else:
        lung_adata.var.loc[i, 'gene'] = i

lung_adata.var.index = lung_adata.var['gene']

In [24]:
# ensamble created some duplicate genes, which we will remove
seen = set()
dupes = []

for x in lung_adata.var.index:
    if x in seen:
        dupes.append(x)
    else:
        seen.add(x)

In [26]:
# Drop the genes that aren't in the SCT mtx, and add SCT to a layer
# This kicks out about ~11 genes
print(lung_adata.var.index)
gene_indicator = np.in1d(lung_adata.var_names, dupes)
lung_adata = lung_adata[:, ~gene_indicator]
print(lung_adata.var.index)

Index(['Xkr4', 'Gm1992', 'Gm19938', 'Sox17', 'Mrpl15', 'ENSMUSG00000120403',
       'Lypla1', 'Tcea1', 'Rgs20', 'Atp6v1h',
       ...
       '1700111N16Rik', 'Zfp92', 'Gm27000', 'Gpr165', 'Rtl9', 'Gm15097',
       'Gm15091', 'Klhl34', 'Gm21454', 'ENSMUSG00000121343'],
      dtype='object', name='gene', length=23663)
Index(['Xkr4', 'Gm1992', 'Gm19938', 'Sox17', 'Mrpl15', 'ENSMUSG00000120403',
       'Lypla1', 'Tcea1', 'Rgs20', 'Atp6v1h',
       ...
       '1700111N16Rik', 'Zfp92', 'Gm27000', 'Gpr165', 'Rtl9', 'Gm15097',
       'Gm15091', 'Klhl34', 'Gm21454', 'ENSMUSG00000121343'],
      dtype='object', name='gene', length=23641)


# Merge and annotate clusters

In [36]:
# Add the spatial coordinates
lung_adata.obs['x_loc'] = -10
lung_adata.obs['y_loc'] = -10

for i in lung_adata.obs.index:
    if lung_adata.obs.loc[i, 'tech'] == 'DBiT_seq':
        lung_adata.obs.loc[i, 'y_loc'] = int(i.split('_')[0].split('x')[0])
        lung_adata.obs.loc[i, 'x_loc'] = int(i.split('_')[0].split('x')[1])

In [37]:
# Add timepoint information
time_dict = {'E12_DevCell': 'E12.5',
             'E11_iScience_R1': 'E11.5',
             'E12_Science': 'E12.5',
             'E12_CellReports_R1': 'E12.5',
             'E13_CellReports': 'E13.5',
             'E11lungR1': 'E11.5',
             'E11lungR2': 'E11.5',
             'E11lungR3': 'E11.5',
             'E11lungR4': 'E11.5',
             'E11lungR5': 'E11.5',
             'E12lungR1': 'E12.5',
             'E12lungR2': 'E12.5',
             'E12lungR3': 'E12.5',
             'E12lungR4': 'E12.5',
             'E12lungR5': 'E12.5',
             'E13lungR1': 'E13.5',
             'E13lungR2': 'E13.5',
             'E13lungR3': 'E13.5',
             'E13lungR4': 'E13.5'}

# add a timepoint to the dataset
lung_adata.obs['timepoint'] = ''
for i in lung_adata.obs.index:
    lung_adata.obs.loc[i, 'timepoint'] = time_dict[lung_adata.obs.loc[i, 'orig.ident']]

In [38]:
seurat_cluster_colors_list = lung_adata.uns['seurat_clusters_colors']
clusters = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18']

seurat_colors = {}
for i in range(0, len(seurat_cluster_colors_list)):
    seurat_colors[str(i)] = seurat_cluster_colors_list[i]

for cluster in clusters:
    plot_spatial_groups(adata = lung_adata[lung_adata.obs['seurat_clusters'] == cluster],
                        meta_col = 'seurat_clusters', 
                        background_adata = lung_adata,
                        show = False,
                        output_loc = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/spatial_clustering/clusters/',
                        name = '{}_spatial_organization.pdf'.format(cluster))

### Annotate Clusters:
- 1 - DistalEpithelial
- 9 - ProximalEpithelial
- 0, 5, 2 - MesenchymeProgenitor
- 3, 10 - SyntheticMesenchyme
- 4 - SmoothMuscleProgenitor
- 8 - Smooth Muscle
- 7 - Mesothelium
- 6, 18 - VascularEndothelial
- 12 - Immune Cells
- 11 - Precartilage
- 15 - Pericyte
- 17 - Heart
- 16 - Neuron
- 13 - Blood
- 14 - DbitEpithelialBridge

In [2]:
assignment_dict = {'0': 'MesenchymeProgenitor',
                   '1': 'Epithelial(Distal)',
                   '2': 'MesenchymeProgenitor',
                   '3': 'SyntheticMesenchyme',
                   '4': 'SmoothMuscleProgenitor',
                   '5': 'MesenchymeProgenitor',
                   '6': 'VascularEndothelial',
                   '7': 'Mesothelium',
                   '8': 'SmoothMuscle',
                   '9': 'Epithelial(Proximal)',
                   '10': 'SyntheticMesenchyme',
                   '11': 'Precartilage',
                   '12': 'Immune',
                   '13': 'Blood',
                   '14': 'DbitEpithelialBridge',
                   '15': 'Pericyte',
                   '16': 'Neuron',
                   '17': 'Heart',
                   '18': 'VascularEndothelial'}

In [49]:
cellstate_list = [assignment_dict[i] for i in lung_adata.obs['seurat_clusters']]

lung_adata.obs['cellstate'] = cellstate_list

In [7]:
lung_adata.obs['cellstate'] = lung_adata.obs['cellstate'].astype('str')
for i in lung_adata.obs.index:
    if lung_adata.obs.loc[i, 'cellstate'] in ['Epithelial(Distal)', 'Epithelial(Proximal)']:
        lung_adata.obs.loc[i, 'cellstate'] = 'Epithelial'

In [55]:
for i in lung_adata.obs['cellstate'].unique():
    test_df = pd.DataFrame(index = lung_adata.uns['rank_genes_groups']['names'][i])

    col_names = ['scores', 'logfoldchanges', 'pvals', 'pvals_adj', 'pts', 'pts_rest']

    for col in col_names:
        test_df[col] = lung_adata.uns['rank_genes_groups'][col][i]
        
    genes = test_df.index
    test_df['{}_avg_expr'.format(i)] = lung_adata[lung_adata.obs['cellstate'] == i][:, genes].X.mean(axis = 0).tolist()[0]
    test_df['not_{}_avg_expr'.format(i)] = lung_adata[lung_adata.obs['cellstate'] != i][:, genes].X.mean(axis = 0).tolist()[0]
    test_df['manualLogFC'] = np.log2(test_df['{}_avg_expr'.format(i)] + 0.01) - np.log2(test_df['not_{}_avg_expr'.format(i)] + 0.01)
    
    test_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/marker_genes/cellstates/{}_vs_rest.csv'.format(i))

# Calculate Cell cycle scores

In [68]:
s_genes_temp =  ["MCM5", "PCNA", "TYMS", "FEN1", "MCM2", "MCM4", "RRM1", "UNG", "GINS2", "MCM6", "CDCA7", "DTL", "PRIM1", "UHRF1", 
            "MLF1IP", "HELLS", "RFC2", "RPA2", "NASP", "RAD51AP1", "GMNN", "WDR76", "SLBP", "CCNE2", "UBR7", "POLD3", "MSH2",
            "ATAD2", "RAD51", "RRM2", "CDC45", "CDC6", "EXO1", "TIPIN", "DSCC1", "BLM", "CASP8AP2", "USP1", "CLSPN", "POLA1",
            "CHAF1B", "BRIP1", "E2F8"]
g2m_genes_temp = ["HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A", "NDC80", "CKS2", "NUF2", "CKS1B", "MKI67", "TMPO",
             "CENPF", "TACC3", "FAM64A", "SMC4", "CCNB2", "CKAP2L", "CKAP2", "AURKB", "BUB1", "KIF11", "ANP32E", "TUBB4B",
             "GTSE1", "KIF20B", "HJURP", "CDCA3", "HN1", "CDC20", "TTK", "CDC25C", "KIF2C", "RANGAP1", "NCAPD2", "DLGAP5",
             "CDCA2", "CDCA8", "ECT2", "KIF23", "HMMR", "AURKA", "PSRC1", "ANLN", "LBR", "CKAP5", "CENPE", "CTCF", "NEK2",
             "G2E3", "GAS2L3", "CBX5", "CENPA" ]

# convert the gene names to the right format
s_genes = []
g2m_genes = []

for i in s_genes_temp:
    temp_i = i[0].upper()
    for j in i[1:]:
        temp_i = temp_i + j.lower()
        
    s_genes.append(temp_i)
    
for i in g2m_genes_temp:
    temp_i = i[0].upper()
    for j in i[1:]:
        temp_i = temp_i + j.lower()
        
    g2m_genes.append(temp_i)
    
sc.tl.score_genes_cell_cycle(lung_adata, s_genes = s_genes, g2m_genes = g2m_genes)

In [70]:
state_phase_proportions = pd.DataFrame(index = cellstates, columns = ['G1', 'G2M', 'S'])
for state in cellstates:
    total = len(lung_adata.obs[lung_adata.obs['cellstate'] == state])
    
    state_phase_proportions.loc[state, 'G1'] = len(lung_adata.obs[(lung_adata.obs['phase'] == 'G1') & (lung_adata.obs['cellstate'] == state)]) / total
    state_phase_proportions.loc[state, 'G2M'] = len(lung_adata.obs[(lung_adata.obs['phase'] == 'G2M') & (lung_adata.obs['cellstate'] == state)]) / total
    state_phase_proportions.loc[state, 'S'] = len(lung_adata.obs[(lung_adata.obs['phase'] == 'S') & (lung_adata.obs['cellstate'] == state)]) / total

In [71]:
state_phase_proportions

,G1,G2M,S
MesenchymeProgenitor,0.072471,0.570289,0.35724
SyntheticMesenchyme,0.330036,0.466957,0.203008
SmoothMuscleProgenitor,0.219178,0.372907,0.407915
SmoothMuscle,0.423469,0.414286,0.162245
Mesothelium,0.273869,0.479899,0.246231
Epithelial(Distal),0.033552,0.634816,0.331631
Epithelial(Proximal),0.135269,0.609065,0.255666
VascularEndothelial,0.090307,0.706383,0.20331
Immune,0.187234,0.553191,0.259574
Pericyte,0.172142,0.525624,0.302234


# Add position annotations

In [2]:
file_list = os.listdir('/Genomics/chanlab/blaw/Spatial_Lung/metadata/spatial_annotation_positions/')
file_dict = {'20230306_E12': 'E12lungR1',
             '20230513_E11': 'E11lungR1',
             '20230830_E12': 'E12lungR2',
             '20230920_E12': 'E12lungR3',
             '20231102_E13': 'E13lungR1',
             '20231103_E13': 'E13lungR2',
             '20231120_E11': 'E11lungR2',
             '20231125_E11': 'E11lungR3',
             '20231129_E11': 'E11lungR4',
             '20231230_E13': 'E13lungR3',
             '20240101_E12': 'E12lungR4',
             '20240119_E11': 'E11lungR5'}

suffix_dict = {'E11lungR1': '_2',
               'E11lungR2': '_3',
               'E11lungR3': '_4',
               'E11lungR4': '_5',
               'E11lungR5': '_6',
               'E12lungR2': '_7',
               'E12lungR3': '_8',
               'E12lungR4': '_9',
               'E12lungR5': '_10',
               'E13lungR1': '_11',
               'E13lungR2': '_12',
               'E13lungR3': '_13',
               'E13lungR4': '_14'}

In [74]:
# make a table of all the annotations from pengfei's files
spatial_annotations_df = pd.DataFrame('None', index = lung_adata.obs[lung_adata.obs['tech'] == 'DBiT_seq'].index, columns = ['lobe_trachea', 'left_right', 'clean', 'dataset'])

for i in spatial_annotations_df.index:
    spatial_annotations_df.loc[i, 'dataset'] = lung_adata.obs.loc[i, 'orig.ident']

# Assign the cell types to the dataframe
# The annotations were done on the raw slices before any thresholding, so some cells are not in the integrated object
# There are some cells between the dividing lines that were not annotated as either
for file in file_list:
    exp = '{}_{}'.format(file.split('_')[1], file.split('_')[2])
    ident = file_dict[exp]
    suffix = suffix_dict[ident]
    file_type = file.split('_')[3].split('.')[0]
    
    file_loc = open("/Genomics/chanlab/blaw/Spatial_Lung/metadata/spatial_annotation_positions/{}".format(file), "r")
    cells = [i + suffix for i in list(csv.reader(file_loc, delimiter=","))[0][1:]]
    
    if file_type == 'right':
        for cell in cells:
            if cell in spatial_annotations_df.index:
                spatial_annotations_df.loc[cell, 'left_right'] = 'right'
    elif file_type == 'left':
        for cell in cells:
            if cell in spatial_annotations_df.index:
                spatial_annotations_df.loc[cell, 'left_right'] = 'left'
    elif file_type == 'lobes':
        for cell in cells:
            if cell in spatial_annotations_df.index:
                spatial_annotations_df.loc[cell, 'lobe_trachea'] = 'lobe'
    elif file_type == 'trachea':
        for cell in cells:
            if cell in spatial_annotations_df.index:
                spatial_annotations_df.loc[cell, 'lobe_trachea'] = 'trachea'
    elif file_type == 'clean':
        for cell in cells:
            if cell in spatial_annotations_df.index:
                spatial_annotations_df.loc[cell, 'clean'] = 'clean'

In [75]:
# add the metadata to the lung adata object
lung_adata.obs['lobe_trachea'] = ''
lung_adata.obs['left_right'] = ''
lung_adata.obs['clean'] = ''

for i in lung_adata.obs.index:
    if i in spatial_annotations_df.index:
        if lung_adata.obs.loc[i, 'orig.ident'] == 'E12lungR5' or lung_adata.obs.loc[i, 'orig.ident'] == 'E13lungR4':
            lung_adata.obs.loc[i, 'clean'] = 'unassigned'
            lung_adata.obs.loc[i, 'lobe_trachea'] = 'unassigned'
            lung_adata.obs.loc[i, 'left_right'] = 'unassigned'
        
        elif lung_adata.obs.loc[i, 'orig.ident'] == 'E11lungR5':
            lung_adata.obs.loc[i, 'clean'] = 'clean'
            lung_adata.obs.loc[i, 'lobe_trachea'] = spatial_annotations_df.loc[i, 'lobe_trachea']
            lung_adata.obs.loc[i, 'left_right'] = spatial_annotations_df.loc[i, 'left_right']
        else:
            lung_adata.obs.loc[i, 'lobe_trachea'] = spatial_annotations_df.loc[i, 'lobe_trachea']
            lung_adata.obs.loc[i, 'left_right'] = spatial_annotations_df.loc[i, 'left_right']
            lung_adata.obs.loc[i, 'clean'] = spatial_annotations_df.loc[i, 'clean']
        
    else:
        lung_adata.obs.loc[i, 'lobe_trachea'] = 'scRNA'
        lung_adata.obs.loc[i, 'left_right'] = 'scRNA'
        lung_adata.obs.loc[i, 'clean'] = 'scRNA'
        


In [80]:
lung_adata.write_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_dbit_srf_CellReports_Science_with_pseudotime.h5ad')

lung_adata.obs.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_dbit_srf_CellReports_Science_with_pseudotime_obs.csv')

# Separate branching regions from brightfield annotations

- Pengfei's annotation are present in /Genomics/chanlab/blaw/Spatial_Lung/metadata/branching_annotations/Figure S6_annotate branches.png

In [4]:
suffix_dict = {'E11lungR1': '_2',
               'E11lungR2': '_3',
               'E11lungR3': '_4',
               'E11lungR4': '_5',
               'E11lungR5': '_6',
               'E12lungR2': '_7',
               'E12lungR3': '_8',
               'E12lungR4': '_9',
               'E12lungR5': '_10',
               'E13lungR1': '_11',
               'E13lungR2': '_12',
               'E13lungR3': '_13',
               'E13lungR4': '_14'}

In [5]:
lung_adata = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_dbit_srf_CellReports_Science_with_pseudotime.h5ad')

In [5]:
for dataset in ['E11lungR1', 'E11lungR2', 'E11lungR3', 'E11lungR4', 'E11lungR5', 'E12lungR2', 'E12lungR3', 'E12lungR4', 'E12lungR5']:
    temp_cells = [i.split('_')[0] for i in lung_adata.obs[lung_adata.obs['orig.ident'] == dataset].index]
    
    temp_inds = []
    for y in range(0, 51):
        for x in range(0, 51):
            if '{}x{}'.format(y, x) in temp_cells:
                temp_inds.append('{}x{}'.format(y, x))
    
    temp_df = pd.DataFrame(index = temp_inds, columns = ['cellID', 'y', 'x', 'annotation'])
    
    for i in temp_df.index:
        temp_df.loc[i, 'cellID'] = i + suffix_dict[dataset]
        temp_df.loc[i, 'y'] = int(i.split('x')[0])
        temp_df.loc[i, 'x'] = int(i.split('x')[1])
    
    temp_df.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/metadata/branching_annotation/{}.csv'.format(dataset))

In [6]:
lung_adata.obs['branching_annotation'] = 'Unassigned'

for dataset in ['E11lungR1', 'E11lungR2', 'E11lungR3', 'E11lungR4', 'E11lungR5', 'E12lungR2', 'E12lungR3', 'E12lungR4', 'E12lungR5']:
    temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/metadata/branching_annotation/annotated/{}.csv'.format(dataset))
    temp_df.index = temp_df['cellID']

    for i in temp_df.index:
        lung_adata.obs.loc[i, 'branching_annotation'] = temp_df.loc[i, 'annotation']

In [7]:
lung_adata.obs['tech'].unique()

['scRNA_seq', 'DBiT_seq']
Categories (2, object): ['DBiT_seq', 'scRNA_seq']

In [11]:
for state in lung_adata.obs['cellstate'].unique():
    print(state)
    print('\t', 'Not_Branching', len(lung_adata.obs[(lung_adata.obs['branching_annotation'] == 'Not_Branching') & (lung_adata.obs['cellstate'] == state) & (lung_adata.obs['tech'] == 'DBiT_seq') & (lung_adata.obs['timepoint'] != 'E13.5')]))
    print('\t', 'Branching', len(lung_adata.obs[(lung_adata.obs['branching_annotation'] == 'Branching') & (lung_adata.obs['cellstate'] == state) & (lung_adata.obs['tech'] == 'DBiT_seq') & (lung_adata.obs['timepoint'] != 'E13.5')]))

SyntheticMesenchyme
	 Not_Branching 2271
	 Branching 15
MesenchymeProgenitor
	 Not_Branching 1502
	 Branching 168
SmoothMuscle
	 Not_Branching 483
	 Branching 8
Pericyte
	 Not_Branching 114
	 Branching 1
SmoothMuscleProgenitor
	 Not_Branching 528
	 Branching 24
VascularEndothelial
	 Not_Branching 792
	 Branching 35
Epithelial(Distal)
	 Not_Branching 1359
	 Branching 180
Immune
	 Not_Branching 267
	 Branching 8
Precartilage
	 Not_Branching 404
	 Branching 3
Blood
	 Not_Branching 357
	 Branching 13
Mesothelium
	 Not_Branching 578
	 Branching 53
Neuron
	 Not_Branching 365
	 Branching 2
Epithelial(Proximal)
	 Not_Branching 574
	 Branching 4
DbitEpithelialBridge
	 Not_Branching 362
	 Branching 25
Heart
	 Not_Branching 1
	 Branching 0


In [12]:
lung_adata.write_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_dbit_srf_CellReports_Science_with_pseudotime.h5ad')

# Look at the CARD proportion results

In [6]:
lung_adata.obs['max_prop'] = 0
lung_adata.obs['max_cluster'] = 'NA'

for exp in lung_adata.obs[lung_adata.obs['tech'] == 'DBiT_seq']['orig.ident'].unique():
    card_proportions = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/CARD/{}_CARD_proportions.csv'.format(exp), index_col = 0)
    break
    for cell in card_proportions.index:
        lung_adata.obs.loc[cell, 'max_prop'] = card_proportions.loc[cell].max()
        lung_adata.obs.loc[cell, 'max_cluster'] = card_proportions.loc[cell].idxmax(axis=0)

In [109]:
lung_adata.write_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_dbit_srf_CellReports_Science_with_pseudotime.h5ad')

In [110]:
lung_adata.obs.to_csv('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_dbit_srf_CellReports_Science_with_pseudotime_obs.csv')

# Read in the epithelial subcluster

In [81]:
lung_adata = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_dbit_srf_CellReports_Science_with_pseudotime.h5ad')
epi_subcluster = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/integrated/subclusters/Lung_integrated_epithelial_subset.h5ad')

In [83]:
# update the subcluster object
epi_adata = lung_adata[epi_subcluster.obs.index,:].copy()
epi_adata.obs = epi_subcluster.obs.copy()
epi_adata.uns = epi_subcluster.uns
epi_adata.obsm = epi_subcluster.obsm
epi_adata.obsp = epi_subcluster.obsp
    
# Add metadata from the lung_adata obs table to the epi_subcluster one
epi_adata.obs['timepoint'] = lung_adata[epi_subcluster.obs.index,:].obs['timepoint']
epi_adata.obs['S_score'] = lung_adata[epi_subcluster.obs.index,:].obs['S_score']
epi_adata.obs['G2M_score'] = lung_adata[epi_subcluster.obs.index,:].obs['G2M_score']
epi_adata.obs['phase'] = lung_adata[epi_subcluster.obs.index,:].obs['phase']
epi_adata.obs['lobe_trachea'] = lung_adata[epi_subcluster.obs.index,:].obs['lobe_trachea']
epi_adata.obs['left_right'] = lung_adata[epi_subcluster.obs.index,:].obs['left_right']
#epi_adata.obs['max_prop'] = lung_adata[epi_subcluster.obs.index,:].obs['max_prop']
epi_adata.obs['x_loc'] = lung_adata[epi_subcluster.obs.index,:].obs['x_loc']
epi_adata.obs['y_loc'] = lung_adata[epi_subcluster.obs.index,:].obs['y_loc']
epi_adata.obs['tech'] = lung_adata[epi_subcluster.obs.index,:].obs['tech']

In [86]:
subcluster_new_names = {'0': 'Proximal',
                        '1': 'Distal',
                        '2': 'DistalIntermediate',
                        '3': 'Distal',
                        '4': 'DistalIntermediate',
                        '5': 'ProximalIntermediate',
                        '6': 'ProximalIntermediate'}

In [87]:
epi_adata.obs['subcluster_cellstate'] = ''
epi_adata.obs['cellstate'] = ''

for i in epi_adata.obs.index:
    epi_adata.obs.loc[i, 'subcluster_cellstate'] = subcluster_new_names[epi_adata.obs.loc[i, 'seurat_clusters']]
    epi_adata.obs.loc[i, 'cellstate'] = lung_adata.obs.loc[i, 'cellstate']

In [94]:
subcluster_pos = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/metadata/pseudotime/epithelial_subcluster/Lung_epithelial_subcluster_integrated_monocle_pos.txt', sep = '\t', index_col = 0)
scRNA_pos = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/metadata/pseudotime/epithelial_subcluster/Lung_epithelial_subcluster_scRNA_only_integrated_monocle_pos.txt', sep = '\t', index_col = 0)

epi_adata.obs['scRNA_only_pseudotime'] = -100
epi_adata.obs['epithelial_pseudotime'] = -100

for i in subcluster_pos.index:
    epi_adata.obs.loc[i, 'epithelial_pseudotime'] = subcluster_pos.loc[i, 'x']

for i in scRNA_pos.index:
    epi_adata.obs.loc[i, 'scRNA_only_pseudotime'] = scRNA_pos.loc[i, 'x']

In [95]:
epi_adata.write_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_epithelial_subcluster_pseudotime.h5ad')

# Look at the mesenchyme subcluster

In [4]:
lung_adata = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_dbit_srf_CellReports_Science_with_pseudotime.h5ad')
#mes_subcluster = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/integrated/subclusters/Lung_integrated_mesenchyme_subset.h5ad')

In [9]:
# update the subcluster object
mes_adata = lung_adata[mes_subcluster.obs.index,:].copy()
mes_adata.obs = mes_subcluster.obs.copy()
mes_adata.uns = mes_subcluster.uns
mes_adata.obsm = mes_subcluster.obsm
mes_adata.obsp = mes_subcluster.obsp
    
# Add metadata from the lung_adata obs table to the epi_subcluster one
mes_adata.obs['timepoint'] = lung_adata[mes_subcluster.obs.index,:].obs['timepoint']
mes_adata.obs['S_score'] = lung_adata[mes_subcluster.obs.index,:].obs['S_score']
mes_adata.obs['G2M_score'] = lung_adata[mes_subcluster.obs.index,:].obs['G2M_score']
mes_adata.obs['phase'] = lung_adata[mes_subcluster.obs.index,:].obs['phase']
mes_adata.obs['lobe_trachea'] = lung_adata[mes_subcluster.obs.index,:].obs['lobe_trachea']
mes_adata.obs['left_right'] = lung_adata[mes_subcluster.obs.index,:].obs['left_right']
#mes_adata.obs['max_prop'] = lung_adata[mes_subcluster.obs.index,:].obs['max_prop']
mes_adata.obs['x_loc'] = lung_adata[mes_subcluster.obs.index,:].obs['x_loc']
mes_adata.obs['y_loc'] = lung_adata[mes_subcluster.obs.index,:].obs['y_loc']
mes_adata.obs['tech'] = lung_adata[mes_subcluster.obs.index,:].obs['tech']
mes_adata.obs['cellstate'] = lung_adata[mes_subcluster.obs.index,:].obs['cellstate']
mes_adata.obs['smooth_muscle_pseudotime'] = lung_adata[mes_subcluster.obs.index,:].obs['smooth_muscle_pseudotime']
mes_adata.obs['synthetic_mesenchyme_pseudotime'] = lung_adata[mes_subcluster.obs.index,:].obs['synthetic_mesenchyme_pseudotime']

In [10]:
sm_subcluster_pos = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/metadata/pseudotime/smooth_muscle_subcluster/Lung_smooth_muscle_subcluster_integrated_monocle_pos.txt', sep = '\t', index_col = 0)
syn_subcluster_pos = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/metadata/pseudotime/synthetic_mesenchyme_subcluster/Lung_synthetic_mesenchyme_subcluster_integrated_monocle_pos.txt', sep = '\t', index_col = 0)
mes_subcluster_pos = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/metadata/pseudotime/mesenchyme_subcluster/Lung_mesenchyme_subcluster_integrated_monocle_pos.txt', sep = '\t', index_col = 0)

mes_adata.obs['smooth_muscle_subcluster_pseudotime'] = -100
mes_adata.obs['synthetic_mesenchyme_subcluster_pseudotime'] = -100
mes_adata.obs['mesenchyme_subcluster_pseudotime'] = -100

for i in sm_subcluster_pos.index:
    mes_adata.obs.loc[i, 'smooth_muscle_subcluster_pseudotime'] = sm_subcluster_pos.loc[i, 'x']

for i in syn_subcluster_pos.index:
    mes_adata.obs.loc[i, 'synthetic_mesenchyme_subcluster_pseudotime'] = syn_subcluster_pos.loc[i, 'x']

for i in mes_subcluster_pos.index:
    mes_adata.obs.loc[i, 'mesenchyme_subcluster_pseudotime'] = mes_subcluster_pos.loc[i, 'x']

In [12]:
subcluster_new_names = {'0': 'MP1',
                        '1': 'MatrixMesenchymeProgenitor',
                        '2': 'MP1',
                        '3': 'SmoothMuscleProgenitor',
                        '4': 'MP2',
                        '5': 'MP2',
                        '6': 'MP3',
                        '7': 'SmoothMuscle',
                        '8': 'MatrixMesenchyme',
                        '9': 'MP1',
                        '10': 'Precartilage',
                        '11':'Pericyte'}

In [15]:
mes_adata.obs['subcluster_cellstate'] = ''
mes_adata.obs['cellstate'] = ''

for i in mes_adata.obs.index:
    mes_adata.obs.loc[i, 'subcluster_cellstate'] = subcluster_new_names[mes_adata.obs.loc[i, 'seurat_clusters']]
    mes_adata.obs.loc[i, 'cellstate'] = lung_adata.obs.loc[i, 'cellstate']

In [13]:
mes_adata.write_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_mesenchyme_subcluster_pseudotime.h5ad')